In [1]:
from math import ceil
from pathlib import Path
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

FRACTION = 0.07
DUPLICATES_FRACTION = 0.01
INPUT_PATH = '/opt/spark/landing/taxi/yellow_tripdata_2025-01.parquet'
DUPLICATE_FILE = '/opt/spark/landing/taxi/yellow_tripdata_2024-03.parquet'
TMP_PATH ='/opt/spark/landing/taxi/ext'
OUTPUT_FILE ='/opt/spark/landing/taxi/yellow_tripdata_2025-01_ext.parquet'
# TEMP_OUTPUT_PATH = OUTPUT_FILE.with_name('_air_quality_2025_update_tmp')\
spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('GenerateTaxiTrips2025')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 14:09:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
original_count = spark.sql("""select count(1) as cnt from delta.`s3a://delta-data/gold/gold_integrated_taxi_trips`""").first()['cnt']


26/09/27 14:09:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/27 14:09:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
df = spark.read.parquet(INPUT_PATH).limit(int(original_count*FRACTION))
df_dup =spark.read.parquet(DUPLICATE_FILE).limit(int(DUPLICATES_FRACTION*original_count)).withColumn(
    "cbd_congestion_fee",
    F.lit(None).cast("double")
)
df_new = df.unionByName(df_dup)
df = df.drop("cbd_congestion_fee")

In [4]:
import glob
import os
import shutil

df.coalesce(1).write.mode("overwrite").parquet(TMP_PATH)
part_file = glob.glob(os.path.join(TMP_PATH, "part-*.parquet"))[0]
shutil.move(part_file, OUTPUT_FILE)
shutil.rmtree(TMP_PATH)